# **Préparation d’un Dataset de Distillation Financière pour le Fine-Tuning d’un Small Language Model**

### Master Économétrie & Statistiques

Ashot AKOPOV — Henri BARDONNAUT — Tristan CHAMBERT

## Objectif du notebook

**Master Économétrie & Statistiques**  
**Ashot AKOPOV — Henri BARDONNAUT — Tristan CHAMBERT**

---

## Résumé

Ce notebook constitue la première brique du pipeline de fine-tuning. Son objectif est de transformer un dataset brut de distillation financière en jeux de données propres, auditables et directement exploitables pour un entraînement SFT/QLoRA.

La logique suivie est la suivante :

1. charger le dataset source ;
2. supprimer les observations non exploitables ;
3. filtrer les données pour conserver les exemples pertinents pour la finance et le raisonnement quantitatif ;
4. analyser les traces de raisonnement générées par le modèle teacher ;
5. construire trois jeux de données finaux :
   - `reasoning_train` ;
   - `response_train` ;
   - `test` ;
6. exporter les fichiers au format `.jsonl` pour l’entraînement et `.parquet` pour l’analyse.

# 1. Import des librairies

In [ ]:
import re
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# 2. Chargement et préparation des données

Nous chargeons le dataset distillé de DeepSeek-R1 spécialisé en finance afin d’effectuer les étapes de nettoyage, de filtrage et de structuration nécessaires au pipeline de fine-tuning.

### Finance_DeepSeek-R1-Distill-dataset

Le dataset utilisé provient du repository Hugging Face `heladell/Finance_DeepSeek-R1-Distill-dataset`. Il contient des exemples de questions/réponses financières ainsi que des traces de raisonnement (`<think>`) issues d’un modèle teacher de type DeepSeek-R1.

In [ ]:
dataset = load_dataset(
    "heladell/Finance_DeepSeek-R1-Distill-dataset",
    split="train"
)

In [ ]:
df = dataset.to_pandas()

# 3. Analyse exploratoire des données

## 3.1 Structure et dimensions

Cette étape permet d’inspecter les colonnes disponibles et la taille globale du dataset.

In [ ]:
df.columns

Index(['conversation_id', 'instruction', 'response', 'conversations',
       'gen_input_configs', 'gen_response_configs', 'intent', 'knowledge',
       'difficulty', 'difficulty_generator', 'input_quality',
       'quality_explanation', 'quality_generator', 'task_category',
       'other_task_category', 'task_category_generator', 'language'],
      dtype='object')

In [ ]:
df.shape

(26952, 17)

## 3.2 Contrôle des réponses manquantes

Le fine-tuning supervisé nécessite une paire instruction/réponse complète. Les observations sans réponse ne peuvent donc pas être utilisées pour l’entraînement et sont supprimées.

In [ ]:
has_response = df["response"].notna().sum()
has_response

np.int64(23979)

In [ ]:
df = df[df['response'].notna()]

## 3.3 Distribution des catégories de tâches

Le dataset contient plusieurs types de tâches. Certaines catégories, comme l’écriture créative ou le jeu de rôle, sont peu pertinentes pour un modèle spécialisé en finance quantitative. Cette analyse permet d’identifier les catégories à conserver ou à exclure.

In [ ]:
per_task = df["task_category"].value_counts().to_frame().reset_index()
per_task

,task_category,count
0,Math,20397
1,Data analysis,1040
2,Information seeking,996
3,Reasoning,606
4,Planning,444
5,Coding & Debugging,390
6,Advice seeking,64
7,Editing,30
8,Brainstorming,7
9,Role playing,1


In [ ]:
nus_cat = ("Coding & Debugging", "Advice seeking", "Editing", "Brainstorming", "Role playing", "Creative writing")
df = df[~df["task_category"].isin(nus_cat)]

## 3.4 Filtrage linguistique

Afin de maintenir une cohérence linguistique pendant l’entraînement, le dataset est restreint aux exemples anglophones.

Ce choix est cohérent avec le dataset source, le modèle cible et les prompts d’entraînement utilisés dans la suite du projet.

In [ ]:
per_lan = df["language"].value_counts().to_frame().reset_index()
per_lan

,language,count
0,EN,23227
1,LA,173
2,AZ,20
3,DA,18
4,DE,7
5,RO,7
6,CA,4
7,ST,4
8,EO,4
9,YO,4


In [ ]:
df = df[df["language"] == "EN"]

# 4. Filtrage métier des données

Cette section applique des règles de filtrage afin de spécialiser le dataset sur les tâches liées à la finance, aux marchés financiers, à l’actuariat, à la comptabilité et au raisonnement quantitatif.

## 4.1 Détection des contenus financiers

## 4.1 Détection heuristique des contenus financiers

Le filtrage repose sur une liste de mots-clés financiers couvrant plusieurs domaines :

- finance d’entreprise ;
- marchés financiers ;
- banque et assurance ;
- comptabilité ;
- macroéconomie ;
- actuariat.

Cette approche est simple, interprétable et facile à auditer. Elle peut toutefois générer des faux positifs ou des faux négatifs, car elle ne capture pas toute la sémantique du texte.

In [ ]:
def is_finance_related(row):

    finance_keywords = [

        # finance entreprise
        "ebitda",
        "cash flow",
        "free cash flow",
        "fcf",
        "capex",
        "valuation",
        "revenue",
        "profit",
        "margin",
        "dividend",

        # marchés financiers
        "bond",
        "equity",
        "stock",
        "portfolio",
        "asset",
        "liability",
        "yield",
        "volatility",
        "option",
        "derivative",
        "credit",
        "interest rate",
        "inflation",
        "market risk",

        # banque / assurance
        "loan",
        "mortgage",
        "insurance",
        "solvency",
        "scr",
        "actuarial",
        "taeg",

        # comptabilité / analyse
        "balance sheet",
        "income statement",
        "financial statement",
        "return on equity",
        "roe",
        "roa",

        # économie
        "gdp",
        "federal reserve",
        "fed",
        "central bank",
        "monetary policy"
    ]

    text = (
        str(row["instruction"]) + " " +
        str(row["response"]) + " " +
        str(row.get("intent", "")) + " " +
        str(row.get("knowledge", ""))
    ).lower()

    return any(
        keyword in text
        for keyword in finance_keywords
    )


In [ ]:
df["finance_related"] = df.apply(
    is_finance_related,
    axis=1
)

## 4.2 Analyse du filtrage financier

Le tableau croisé ci-dessous permet de mesurer la part des exemples considérés comme financiers selon la catégorie de tâche.

In [ ]:
pd.crosstab(
    df["task_category"],
    df["finance_related"],
    margins=True
)

finance_related,False,True,All
task_category,,,
Data analysis,114,925,1039
Information seeking,162,826,988
Math,5532,14616,20148
Planning,48,395,443
Reasoning,109,497,606
All,5965,17259,23224


## 4.3 Application du filtre métier

Les exemples non liés à la finance sont supprimés afin de spécialiser le dataset.

Exception : les exemples de la catégorie `Math` sont conservés, même lorsqu’ils ne contiennent pas explicitement de mots-clés financiers.  
Ils restent utiles pour renforcer les capacités de raisonnement numérique et symbolique du modèle.

In [ ]:
df = df[
    (df["task_category"] == "Math")
    |
    (df["finance_related"])
]

In [ ]:
df.shape

(22794, 18)

# 5. Analyse des traces de raisonnement (`<think>`)

Cette section analyse les blocs `<think>` produits par le modèle teacher.

Ces traces constituent une supervision intermédiaire utile : elles montrent au modèle étudiant comment structurer un raisonnement avant de produire une réponse finale.

## 5.1 Séparation du raisonnement et de la réponse finale

Les réponses contenant des blocs `<think>` sont séparées en deux parties :

- le raisonnement explicite ;
- la réponse finale visible.

Cette séparation permet de construire deux datasets complémentaires :
- `reasoning_train`, qui conserve les traces de raisonnement ;
- `response_train`, qui ne conserve que les réponses finales concises.

In [ ]:
def has_think(response):

    if response is None:
        return False

    return bool(
        re.search(
            r"<think>.*?</think>",
            str(response),
            flags=re.DOTALL
        )
    )


# Ajout colonne
df["has_think"] = df["response"].apply(has_think)

In [ ]:
df["has_think"].value_counts().to_frame().reset_index()

,has_think,count
0,True,17046
1,False,5748


## 5.3 Évaluation heuristique de la qualité du raisonnement

Toutes les traces de raisonnement ne sont pas nécessairement utiles pour l’entraînement.  
Certaines peuvent être trop courtes, trop longues, bruitées, hésitantes ou faiblement structurées.

Nous définissons donc une fonction de contrôle qualité fondée sur plusieurs critères :

- longueur minimale du raisonnement ;
- présence d’une réponse finale exploitable ;
- ratio entre longueur du raisonnement et longueur de l’instruction ;
- présence de marqueurs de structure logique ;
- présence éventuelle de formules ou de calculs ;
- niveau de bruit ou d’hésitation.

In [ ]:
# =====================================================
# 1) EXTRAIRE LE RAISONNEMENT <think>
# =====================================================

def extract_think(response):
    """
    Récupère uniquement le texte entre <think> et </think>.
    Si aucun <think> n'existe, renvoie une chaîne vide.
    """
    match = re.search(
        r"<think>(.*?)</think>",
        str(response),
        flags=re.DOTALL
    )
    return match.group(1).strip() if match else ""


# =====================================================
# 2) ENLEVER LE <think> POUR GARDER LA REPONSE FINALE
# =====================================================

def remove_think(response):
    """
    Supprime le bloc <think>...</think>.
    Il reste normalement la réponse finale.
    """
    return re.sub(
        r"<think>.*?</think>",
        "",
        str(response),
        flags=re.DOTALL
    ).strip()


# =====================================================
# 3) EVALUER SI LE RAISONNEMENT EST BON
# =====================================================

def is_good_reasoning(row):
    """
    Renvoie True si le raisonnement semble exploitable pour l'entraînement.
    Renvoie False s'il est trop court, trop bruité, disproportionné,
    ou sans vraie réponse finale.
    """

    instruction = str(row["instruction"])
    response = str(row["response"])

    think = extract_think(response)
    final_answer = remove_think(response)

    # -------------------------
    # Longueurs
    # -------------------------

    instruction_words = len(instruction.split())
    think_words = len(think.split())
    final_answer_words = len(final_answer.split())

    # Ratio : taille du raisonnement par rapport à la question
    # max(..., 1) évite une division par zéro
    think_instruction_ratio = think_words / max(instruction_words, 1)

    # -------------------------
    # Bruit / hésitations
    # -------------------------

    noise_words = [
        "wait",
        "maybe",
        "hmm",
        "perhaps",
        "let me think",
        "actually"
    ]

    noise_score = sum(
        think.lower().count(word)
        for word in noise_words
    )

    # -------------------------
    # Structure logique
    # -------------------------

    has_structure = any(
        marker in think.lower()
        for marker in [
            "first",
            "then",
            "finally",
            "therefore",
            "step",
            "1.",
            "2.",
            "so,"
        ]
    )

    # -------------------------
    # Présence de formules / calculs
    # -------------------------

    has_formula = any(
        symbol in think
        for symbol in ["=", "+", "-", "/", "*", "%"]
    )

    # -------------------------
    # Réponse finale exploitable
    # -------------------------

    has_final_answer = final_answer_words > 20

    # =================================================
    # REGLES DE REJET DIRECT
    # =================================================

    # Pas de raisonnement
    if think_words == 0:
        return False

    # Raisonnement trop court : peu utile pour apprendre à raisonner
    if think_words < 60:
        return False

    # Pas de vraie réponse finale après le think
    if not has_final_answer:
        return False

    # Beaucoup trop d'hésitations
    if noise_score > 12:
        return False

    # Raisonnement extrêmement disproportionné par rapport à la question
    if think_instruction_ratio > 20:
        return False

    # =================================================
    # REGLES SELON LE RATIO THINK / INSTRUCTION
    # =================================================

    # Cas 1 : raisonnement proportionné
    # Exemple : instruction 200 mots, think 400 mots
    if think_instruction_ratio <= 4:
        return noise_score <= 8

    # Cas 2 : raisonnement assez long par rapport à l'instruction
    # On demande au moins une structure OU une formule
    if 4 < think_instruction_ratio <= 10:
        return (
            noise_score <= 5
            and (has_structure or has_formula)
        )

    # Cas 3 : raisonnement très long par rapport à l'instruction
    # On l'accepte seulement s'il est très propre et structuré
    if 10 < think_instruction_ratio <= 20:
        return (
            noise_score <= 2
            and has_structure
            and has_formula
        )

    return False

In [ ]:
df["good_reasoning"] = df.apply(
    is_good_reasoning,
    axis=1
)

In [ ]:
pd.crosstab(
    df["has_think"],
    df["good_reasoning"],
    margins=True
)

good_reasoning,False,True,All
has_think,,,
False,5748,0,5748
True,8999,8047,17046
All,14747,8047,22794


In [ ]:
pd.crosstab(
    df["has_think"],
    df["good_reasoning"],
    margins=True
)

good_reasoning,False,True,All
has_think,,,
False,5748,0,5748
True,8999,8047,17046
All,14747,8047,22794


## 5.4 Validation qualitative

Une validation manuelle est réalisée sur des exemples jugés bons et mauvais par l’heuristique.  
Cette étape permet de vérifier que les règles de filtrage produisent des décisions cohérentes avec l’objectif d’entraînement.

In [ ]:
good_examples = df[
    df["good_reasoning"]
]

good_examples.sample(5)[
    ["instruction", "response"]
]

,instruction,response
18002,"When the principal amount of $2,500 is investe...","<think>\nFirst, I need to identify all the giv..."
13996,Tessa wants to save up money to buy a new comp...,"<think>\nFirst, I need to determine how much m..."
19756,### Synergy Effect of Hydrogen Peroxide Additi...,"<think>\nOkay, so I have this problem about th..."
10998,Oliver has been hired by a bank to look over a...,"<think>\nFirst, I need to determine the number..."
6933,The standard deviation of the sample mean is g...,"<think>\nOkay, so I have this problem about st..."


In [ ]:
bad_examples = df[
    (df["has_think"])
    & (~df["good_reasoning"])
]

bad_examples.sample(5)[
    ["instruction", "response"]
]

,instruction,response
10891,"A convex quadrilateral ABCD, with diagonals AC...","<think>\nOkay, so I have this problem about a ..."
7276,"A firm produces two products, chocolate (\(X\)...","<think>\nOkay, so I have this problem where a ..."
7804,Academic freedom is not an unlimited right in ...,"<think>\nOkay, so I need to understand what th..."
8988,"In the coordinate plane, the point \( (2, 3) \...","<think>\nOkay, so I need to find the equation ..."
12921,"There are 1200 guests at a party. Of them, 200...","<think>\nOkay, so I have this problem here abo..."


# 6. Séparation des réponses et préparation des datasets finaux

À ce stade, le dataset contient les observations filtrées et les indicateurs de qualité.  
Nous allons maintenant créer les colonnes nécessaires pour la stratégie d’entraînement en deux temps.

In [ ]:
def extract_think(response):

    match = re.search(
        r"<think>(.*?)</think>",
        str(response),
        flags=re.DOTALL
    )

    return match.group(1).strip() if match else ""


def remove_think(response):

    return re.sub(
        r"<think>.*?</think>",
        "",
        str(response),
        flags=re.DOTALL
    ).strip()

In [ ]:
df["reasoning"] = df["response"].apply(
    extract_think
)

df["final_answer"] = df["response"].apply(
    remove_think
)

# 7. Construction des jeux d’entraînement

Cette section transforme le dataset nettoyé en trois jeux de données distincts :

- `reasoning_train` : apprentissage des traces de raisonnement ;
- `response_train` : apprentissage des réponses finales concises ;
- `test` : évaluation finale du modèle.

Le split train/test est effectué avant la création des variantes d’entraînement afin d’éviter toute fuite de données entre les jeux.

In [ ]:
df_raw = df.copy()

## 7.1 Création du dataset nettoyé

Nous conservons uniquement les colonnes utiles à l’analyse, à l’audit qualité et à la construction des datasets d’entraînement.

In [ ]:
# Colonnes utiles pour :
# - analyse
# - statistiques
# - filtering
# - quality checks
# - reasoning filtering

clean_columns = [
    "instruction",
    "response",
    "reasoning",
    "final_answer",
    "task_category",
    "difficulty",
    "input_quality",
    "has_think",
    "finance_related",
    "good_reasoning"
]

df_clean = df_raw[clean_columns].copy()

## 7.2 Séparation train/test

Le dataset est séparé en un jeu d’entraînement et un jeu de test.  
Le jeu de test reste unique et non dupliqué afin de mesurer la qualité finale des réponses produites par le modèle.

Nous souhaitons stratifier la séparation selon plusieurs dimensions importantes : présence d’une trace de raisonnement et niveau de difficulté.

Comme `train_test_split` n’accepte qu’une seule colonne de stratification, une variable composite est créée afin de conserver une distribution équilibrée entre les jeux d’entraînement et de test.

In [ ]:
df_clean["difficulty"].value_counts().to_frame().reset_index()

,difficulty,count
0,medium,14482
1,easy,6962
2,hard,1163
3,very easy,160
4,very hard,3


Les modalités extrêmes de difficulté sont regroupées afin de limiter les classes trop rares et de stabiliser la stratification.

In [ ]:
def clean_difficulty(x):

    if x in ["very easy"]:
        return "easy"

    if x in ["very hard"]:
        return "hard"

    return x


df_clean["difficulty_clean"] = (
    df_clean["difficulty"]
    .apply(clean_difficulty)
)

In [ ]:
df_clean["difficulty_clean"].value_counts()

,count
difficulty_clean,
medium,14482
easy,7122
hard,1166


In [ ]:
df_clean["stratify_col"] = (
    df_clean["has_think"].astype(str)
    + "_"
    + df_clean["difficulty_clean"].astype(str)
)

In [ ]:
df_clean["stratify_col"].value_counts()

,count
stratify_col,
True_medium,10138
True_easy,6438
False_medium,4344
False_hard,709
False_easy,684
True_hard,457
True_None,13
False_None,11


In [ ]:
df_train, df_test = train_test_split(
    df_clean,
    test_size=0.1,
    random_state=42,
    stratify=df_clean["stratify_col"]
)

In [ ]:
print(df_train.shape)
print(df_test.shape)

(20514, 12)
(2280, 12)


## 7.3 Format conversationnel d’entraînement

Les exemples sont convertis en format conversationnel compatible avec les modèles instruct de type Qwen.

Deux formats sont définis :

1. un format avec raisonnement, utilisé pour `reasoning_train` ;
2. un format sans raisonnement visible, utilisé pour `response_train` et `test`.

Le format `reasoning_train` conserve la réponse complète issue du dataset source, y compris les éventuelles traces `<think>`.  
Il sert à transmettre au modèle étudiant des schémas de raisonnement financier et mathématique.

In [ ]:
def format_reasoning(row):

    return f"""<|im_start|>system
You are an expert quantitative finance, actuarial science, and mathematical reasoning assistant.

Provide accurate, rigorous, and concise answers.

Use structured step-by-step reasoning only when necessary for solving complex problems.
For quantitative tasks, clearly explain formulas, assumptions, and calculations.

Avoid unnecessary verbosity, repetition, or low-value reasoning.
Focus on clarity, precision, and efficiency.

This example may include reasoning traces to help the model learn how to reason, but the final answer must remain concise.
<|im_end|>

<|im_start|>user
{row["instruction"]}
<|im_end|>

<|im_start|>assistant
{row["response"]}
<|im_end|>
"""

In [ ]:
def format_response(row):

    return f"""<|im_start|>system
You are an expert quantitative finance, actuarial science, and mathematical reasoning assistant.

Provide accurate, rigorous, and concise answers.

Use structured step-by-step reasoning only when necessary for solving complex problems.
For quantitative tasks, clearly explain formulas, assumptions, and calculations.

Avoid unnecessary verbosity, repetition, or low-value reasoning.
Focus on clarity, precision, and efficiency.

Do not expose internal reasoning. Output only the final concise answer.
<|im_end|>

<|im_start|>user
{row["instruction"]}
<|im_end|>

<|im_start|>assistant
{row["final_answer"]}
<|im_end|>
"""

## 7.4 Création des datasets `reasoning_train` et `response_train`

Les deux datasets sont construits à partir du même split d’entraînement, mais avec un objectif différent :

- `reasoning_train` utilise la réponse complète ;
- `response_train` utilise uniquement la réponse finale nettoyée.

In [ ]:
# reasoning_train
reasoning_train = df_train.copy()

reasoning_train["text"] = reasoning_train.apply(
    format_reasoning,
    axis=1
)

# response_train
response_train = df_train.copy()

response_train["text"] = response_train.apply(
    format_response,
    axis=1
)


## 7.5 Création du dataset de test

Le dataset de test utilise uniquement les réponses finales.  
Cela permet d’évaluer le comportement réellement attendu du modèle : produire des réponses correctes, concises et professionnelles, sans afficher de raisonnement interne.

In [ ]:
test = df_test.copy()

test["text"] = test.apply(
    format_response,
    axis=1
)

# 8. Export des artefacts

Les datasets sont exportés sous deux formats complémentaires :

- `.jsonl` : format standard pour les pipelines de fine-tuning ;
- `.parquet` : format optimisé pour l’analyse, l’audit et la reproductibilité.

Cette double exportation permet de séparer clairement les besoins d’entraînement des besoins d’analyse.

Les fichiers `.jsonl` sont destinés à être déposés sur Hugging Face pour l’entraînement.  
Les fichiers `.parquet` sont conservés localement pour inspecter les données, produire des statistiques et documenter le projet.

In [ ]:
# =====================================================
# EXPORT JSONL
# =====================================================

reasoning_train[["text"]].to_json(
    "reasoning_train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

response_train[["text"]].to_json(
    "response_train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

test[["text"]].to_json(
    "test.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)


# =====================================================
# EXPORT PARQUET
# =====================================================

reasoning_train.to_parquet(
    "reasoning_train.parquet",
    index=False
)

response_train.to_parquet(
    "response_train.parquet",
    index=False
)

test.to_parquet(
    "test.parquet",
    index=False
)

Nous allons maintenant tout télécharger

In [ ]:
from google.colab import files

files.download("reasoning_train.jsonl")
files.download("response_train.jsonl")
files.download("test.jsonl")
files.download("reasoning_train.parquet")
files.download("response_train.parquet")
files.download("test.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 9. Conclusion

Ce notebook a permis de construire un pipeline complet de préparation de données destiné au fine-tuning d’un Small Language Model spécialisé en finance.